# 🛰️ MOONSHOT 2048: Native-Resolution Inference & Submission Engine
### IEEE BigData Cup / Kaggle Solar Filament Segmentation Challenge 2026
**Authority:** ChatGPT Master | **Directives:** 17 & 18 | **Executor:** Antigravity  

Strict Host Rules Enforced:
1. Native 2048 Resolution Direct Inference (YOLOv8l-seg)
2. Assert exactly 180 discovered test images before inference and record inference manifest
3. Greedy Pixel-Carve Zero-Overlap Sanitizer (Assert 0 shared pixels per disk)
4. Zero rows emitted for disks with 0 detections (no dummy masks)
5. Valid pycocotools COCO Fortran RLE encoding


In [ ]:
# ==============================================================================
# CELL 2: Setup & Environment
# ==============================================================================
import os, sys
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip -q install ultralytics==8.4.103 pycocotools scikit-learn


In [ ]:
# ==============================================================================
# CELL 3: Locate Dataset & Trained Weights (Assert Exactly 180 Test Images)
# ==============================================================================
from pathlib import Path
import torch

candidate_paths = [
    Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"),
    Path("/kaggle/input/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"),
    Path("/kaggle/input/filament-segmentation-2026"),
    Path("data/MAGFiLO_1.0_Kaggle_2026"),
]
data_dir = next((p for p in candidate_paths if p.exists()), None)
test_dir = data_dir / "test" / "test_images"
print(f"Test Images Directory: {test_dir}")
assert test_dir.exists(), f"FATAL: Test directory not found at {test_dir}"

# Exact 180 images assertion (Directive 18)
test_images = list(test_dir.glob("*.jpeg")) + list(test_dir.glob("*.jpg"))
print(f"Test Images Discovered on Disk: {len(test_images)}")
assert len(test_images) == 180, f"FATAL: Expected exactly 180 test images, found {len(test_images)}!"

# Locate trained weights
weights_candidates = list(Path("/kaggle/input").glob("**/best.pt")) + list(Path(".").glob("**/best.pt"))
print(f"Weights Candidates Found: {weights_candidates}")
assert len(weights_candidates) > 0, "FATAL: Trained best.pt weights not attached!"
model_weights = weights_candidates[0]
print(f"Selected Weights: {model_weights}")


In [ ]:
# ==============================================================================
# CELL 4: Deploy Inference Modules
# ==============================================================================
from pathlib import Path
Path("metrics").mkdir(parents=True, exist_ok=True)
Path("moonshot_2048").mkdir(parents=True, exist_ok=True)

with open("metrics/__init__.py", "w") as f: f.write("")
with open("moonshot_2048/__init__.py", "w") as f: f.write("")

with open("metrics/pq.py", "w", encoding="utf-8") as f: f.write('"""\nKirillov Panoptic Quality (PQ) scorer for solar filament instance segmentation.\n\nPQ = SQ × RQ\nSQ = mean IoU of matched instances (Segmentation Quality)\nRQ = TP / (TP + 0.5·FP + 0.5·FN) (Recognition Quality)\n\nMatching: greedy unique pairing with IoU > 0.5 threshold.\nReference: Kirillov et al., "Panoptic Segmentation", CVPR 2019.\n"""\n\nimport numpy as np\nimport pycocotools.mask as mask_utils\n\n\ndef encode_mask(mask_hw: np.ndarray) -> str:\n    """Encode a 2D binary mask (H, W) to a COCO RLE counts string.\n\n    Always goes through pycocotools Fortran 3D encode.\n    Never returns a hardcoded string.\n\n    Args:\n        mask_hw: uint8 array of shape (H, W) with values 0 or 1.\n\n    Returns:\n        COCO RLE counts string (e.g. for a 2048×2048 zero mask this\n        will be whatever pycocotools produces — currently \'PPPP4\').\n    """\n    h, w = mask_hw.shape[:2]\n    mask_3d = np.asfortranarray(mask_hw.astype(np.uint8)).reshape((h, w, 1))\n    rle = mask_utils.encode(mask_3d)[0]\n    counts = rle["counts"]\n    if isinstance(counts, bytes):\n        counts = counts.decode("utf-8")\n    return counts\n\n\ndef decode_rle(counts_str: str, h: int | tuple = 2048, w: int = 2048) -> np.ndarray:\n    """Decode a COCO RLE counts string back to a 2D binary mask (H, W)."""\n    if isinstance(h, (tuple, list)):\n        h, w = h[0], h[1]\n    rle = {"size": [int(h), int(w)], "counts": counts_str}\n    mask = mask_utils.decode(rle)\n    if mask.ndim == 3:\n        mask = mask[:, :, 0]\n    return mask\n\n\ndef _iou(mask_a: np.ndarray, mask_b: np.ndarray) -> float:\n    """Compute IoU between two binary masks."""\n    intersection = np.logical_and(mask_a, mask_b).sum()\n    union = np.logical_or(mask_a, mask_b).sum()\n    if union == 0:\n        return 0.0\n    return float(intersection) / float(union)\n\n\ndef pq_score(\n    pred_masks: list[np.ndarray],\n    gt_masks: list[np.ndarray],\n    iou_threshold: float = 0.5,\n) -> dict:\n    """Compute Kirillov Panoptic Quality between predicted and GT instances.\n\n    Matching is greedy: sort all (pred, gt) pairs by descending IoU,\n    accept a pair only if IoU > iou_threshold and neither instance is\n    already matched. Each instance can match at most once.\n\n    Args:\n        pred_masks: list of binary (H, W) uint8 arrays, one per predicted instance.\n        gt_masks:   list of binary (H, W) uint8 arrays, one per GT instance.\n        iou_threshold: minimum IoU for a valid match (default 0.5).\n\n    Returns:\n        dict with keys: PQ, SQ, RQ, TP, FP, FN, matched_ious.\n    """\n    n_pred = len(pred_masks)\n    n_gt = len(gt_masks)\n\n    # Edge cases\n    if n_pred == 0 and n_gt == 0:\n        return {"PQ": 1.0, "SQ": 1.0, "RQ": 1.0, "TP": 0, "FP": 0, "FN": 0, "matched_ious": []}\n    if n_pred == 0:\n        return {"PQ": 0.0, "SQ": 0.0, "RQ": 0.0, "TP": 0, "FP": 0, "FN": n_gt, "matched_ious": []}\n    if n_gt == 0:\n        return {"PQ": 0.0, "SQ": 0.0, "RQ": 0.0, "TP": 0, "FP": n_pred, "FN": 0, "matched_ious": []}\n\n    # Compute full IoU matrix\n    iou_matrix = np.zeros((n_pred, n_gt), dtype=np.float64)\n    for i, pm in enumerate(pred_masks):\n        for j, gm in enumerate(gt_masks):\n            iou_matrix[i, j] = _iou(pm, gm)\n\n    # Greedy matching: sort all pairs by descending IoU\n    pairs = []\n    for i in range(n_pred):\n        for j in range(n_gt):\n            if iou_matrix[i, j] > iou_threshold:\n                pairs.append((iou_matrix[i, j], i, j))\n    pairs.sort(key=lambda x: x[0], reverse=True)\n\n    matched_pred = set()\n    matched_gt = set()\n    matched_ious = []\n\n    for iou_val, pi, gi in pairs:\n        if pi in matched_pred or gi in matched_gt:\n            continue\n        matched_pred.add(pi)\n        matched_gt.add(gi)\n        matched_ious.append(iou_val)\n\n    tp = len(matched_ious)\n    fp = n_pred - tp\n    fn = n_gt - tp\n\n    sq = float(np.mean(matched_ious)) if tp > 0 else 0.0\n    rq = tp / (tp + 0.5 * fp + 0.5 * fn) if (tp + fp + fn) > 0 else 0.0\n    pq = sq * rq\n\n    return {\n        "PQ": round(pq, 6),\n        "SQ": round(sq, 6),\n        "RQ": round(rq, 6),\n        "TP": tp,\n        "FP": fp,\n        "FN": fn,\n        "matched_ious": matched_ious,\n    }\n\n\ndef pq_score_multi(\n    pred_masks_list: list[list[np.ndarray]],\n    gt_masks_list: list[list[np.ndarray]],\n    iou_threshold: float = 0.5,\n) -> dict:\n    """Compute mean PQ / SQ / RQ across multiple images.\n\n    Args:\n        pred_masks_list: list of per-image predicted mask lists.\n        gt_masks_list:   list of per-image GT mask lists.\n        iou_threshold: minimum IoU for matching.\n\n    Returns:\n        dict with mean PQ, SQ, RQ, total TP, FP, FN, and per-image results.\n    """\n    assert len(pred_masks_list) == len(gt_masks_list), "Mismatched image count"\n\n    per_image = []\n    total_tp = total_fp = total_fn = 0\n    sum_pq = sum_sq = sum_rq = 0.0\n\n    for preds, gts in zip(pred_masks_list, gt_masks_list):\n        result = pq_score(preds, gts, iou_threshold)\n        per_image.append(result)\n        total_tp += result["TP"]\n        total_fp += result["FP"]\n        total_fn += result["FN"]\n        sum_pq += result["PQ"]\n        sum_sq += result["SQ"]\n        sum_rq += result["RQ"]\n\n    n = len(pred_masks_list)\n    return {\n        "mean_PQ": round(sum_pq / n, 6) if n > 0 else 0.0,\n        "mean_SQ": round(sum_sq / n, 6) if n > 0 else 0.0,\n        "mean_RQ": round(sum_rq / n, 6) if n > 0 else 0.0,\n        "total_TP": total_tp,\n        "total_FP": total_fp,\n        "total_FN": total_fn,\n        "n_images": n,\n        "per_image": per_image,\n    }\n')
with open("moonshot_2048/config.py", "w", encoding="utf-8") as f: f.write('"""\nmoonshot_2048/config.py — Frozen Configurations for Native-2048 Moonshot Pipeline.\n\nAuthority: ChatGPT Master\nDirective: 17 — Native-2048 Moonshot Toward 0.60 PQ\nExecutor: Antigravity\n"""\n\nfrom pathlib import Path\nimport os\nimport torch\n\n# Base paths & environment detection\nKAGGLE = Path("/kaggle/input").exists()\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\nOUT = Path("/kaggle/working") if KAGGLE else PROJECT_ROOT\n\nCANDIDATE_DATA_DIRS = [\n    Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"),\n    Path("/kaggle/input/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"),\n    Path("/kaggle/input/filament-segmentation-2026"),\n    PROJECT_ROOT / "data" / "MAGFiLO_1.0_Kaggle_2026",\n]\n\nMAGFILO_DIR = next((p for p in CANDIDATE_DATA_DIRS if p.exists()), PROJECT_ROOT / "data" / "MAGFiLO_1.0_Kaggle_2026")\nYOLO_DATA_DIR = OUT / "data" / "yolo_native2048"\nRUNS_DIR = OUT / "runs"\nMODELS_DIR = OUT / "models" / "moonshot_2048"\nSUBMISSIONS_DIR = OUT if KAGGLE else (PROJECT_ROOT / "submissions")\n\nMODELS_DIR.mkdir(parents=True, exist_ok=True)\nSUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)\n\n\nclass MoonshotConfig:\n    """Core Native-2048 Moonshot Hyperparameters (Directive 17 / Directive 18)."""\n    PROJECT_ROOT = PROJECT_ROOT\n    OUT = OUT\n    RUNS_DIR = RUNS_DIR\n    MAGFILO_DIR = MAGFILO_DIR\n    YOLO_DATA_DIR = YOLO_DATA_DIR\n\n    # Architecture & Model weights\n    MODEL_V8L = "yolov8l-seg.pt"\n    MODEL_11L = "yolo11l-seg.pt"\n    \n    # Image resolution: native 2048 with approved fallback attempt chain (Directive 18)\n    IMGSZ = 2048\n    FALLBACK_IMGSZ = [2048, 1792, 1536]\n    FALLBACK_ATTEMPTS = [\n        (2048, 2),\n        (2048, 1),\n        (1792, 1),\n        (1536, 1),\n    ]\n    \n    # Training hyperparameters\n    EPOCHS = 60\n    PATIENCE = 15\n    BATCH = 2  # Starting batch for 16 GB Tesla T4\n    DEVICE = 0  # Single GPU for training to avoid notebook DDP hang\n    AMP = True\n    WORKERS = 2 if os.name != "nt" else 0\n    SEED = 42\n    MAX_DET = 100\n    \n    # Conservative augmentations tailored for solar chromosphere morphology\n    DEGREES = 10.0\n    FLIPUD = 0.5\n    FLIPLR = 0.5\n    MOSAIC = 0.0      # Disabled initially per Directive 17 to preserve native solar geometry\n    COPY_PASTE = 0.0  # Disabled initially\n    CLOSE_MOSAIC = 0\n    \n    # Inference defaults (Directive 17 & 0.55 anchor analysis)\n    CONF = 0.30\n    NMS_IOU = 0.00\n    MIN_AREA = 200\n    OVERLAP_MODE = "trim"  # Strictly enforced zero-overlap greedy carve\n    \n    # Geometric disk clipping\n    SOLAR_DISK_R_FRAC = 0.93  # Clipping radius fraction (~952 px radius from disk center)\n    \n    # Output paths\n    SUBMISSION_PATH = (OUT / "submission.csv") if KAGGLE else (SUBMISSIONS_DIR / "moonshot_submission.csv")\n    V8L_RUN_DIR = RUNS_DIR / "moonshot_v8l_2048"\n    V8L_WEIGHTS_PATH = V8L_RUN_DIR / "weights" / "best.pt"\n    V11L_RUN_DIR = RUNS_DIR / "moonshot_v11l_2048"\n    V11L_WEIGHTS_PATH = V11L_RUN_DIR / "weights" / "best.pt"\n\n    @classmethod\n    def resolve_v8l_weights(cls, explicit_path=None) -> Path:\n        """Resolve YOLOv8l-seg weights across /kaggle/input and run directories."""\n        if explicit_path and Path(explicit_path).exists():\n            return Path(explicit_path)\n        if cls.V8L_WEIGHTS_PATH.exists():\n            return cls.V8L_WEIGHTS_PATH\n        # Scan /kaggle/input for any trained moonshot or yolov8l weights\n        if Path("/kaggle/input").exists():\n            candidates = sorted(list(Path("/kaggle/input").glob("**/best.pt")), key=lambda p: p.stat().st_mtime)\n            if candidates:\n                return candidates[-1]\n            # Also check for external hdjojo weights if attached\n            hd_weights = list(Path("/kaggle/input").glob("**/*yolov8l*.pt"))\n            if hd_weights:\n                return hd_weights[0]\n        return cls.V8L_WEIGHTS_PATH\n')
with open("moonshot_2048/predict_native.py", "w", encoding="utf-8") as f: f.write('"""\nmoonshot_2048/predict_native.py — Native-2048 YOLO Inference Engine & Zero-Overlap Sanitizer.\n\nAuthority: ChatGPT Master\nDirective: 17 — Native-2048 Moonshot Toward 0.60 PQ\nExecutor: Antigravity\n\nFrozen Submission & Prediction Truths:\n1. Native Resolution: Predict directly on 2048x2048 images (or fallback 1792/1536).\n2. Host Zero-Overlap Rule: Strictly zero shared pixels between any two masks on the same disk.\n   - Enforce greedy pixel carve: sort instances descending by confidence, then area.\n   - mask[occupied > 0] = 0; drop if post-carve area < min_area.\n   - Per-disk assertion: sum(mask_i & mask_j) == 0 for all i != j.\n3. Zero-Prediction Rule:\n   - Disks with zero predicted filaments emit ZERO rows in submission.csv.\n   - Never emit dummy zero masks (PPP2, PPP8, etc.).\n4. Encoding:\n   - Valid pycocotools COCO Fortran RLE.\n   - Output format: exactly two columns: filament_id,segmentation_rle.\n"""\n\nimport argparse\nimport sys\nfrom pathlib import Path\nfrom typing import Dict, List, Optional, Tuple\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom metrics.pq import encode_mask, decode_rle\nfrom moonshot_2048.config import MoonshotConfig, MAGFILO_DIR, SUBMISSIONS_DIR\n\n\ndef apply_solar_limb_mask(mask: np.ndarray, r_frac: float = MoonshotConfig.SOLAR_DISK_R_FRAC) -> np.ndarray:\n    """Zero out any predicted pixels falling outside the solar chromosphere limb."""\n    h, w = mask.shape[:2]\n    cx, cy = w // 2, h // 2\n    max_r = int((min(w, h) / 2.0) * r_frac)\n    \n    # Fast distance check: zero out outside circle\n    y_indices, x_indices = np.ogrid[:h, :w]\n    dist_from_center_sq = (x_indices - cx)**2 + (y_indices - cy)**2\n    mask[dist_from_center_sq > max_r**2] = 0\n    return mask\n\n\ndef sanitize_instances_zero_overlap(\n    masks: List[np.ndarray],\n    confidences: List[float],\n    min_area: int = MoonshotConfig.MIN_AREA,\n    r_frac: float = MoonshotConfig.SOLAR_DISK_R_FRAC,\n) -> Tuple[List[np.ndarray], List[float]]:\n    """Strict Greedy Pixel-Carve Zero-Overlap Sanitizer.\n    \n    Order: Confidence descending, then area descending.\n    Each mask carves away pixels already claimed by higher-priority masks.\n    Post-carve masks with area < min_area are discarded.\n    Finally, strictly asserts that no two masks share any pixels.\n    """\n    if not masks:\n        return [], []\n\n    # Pair and sort\n    items = []\n    for m, c in zip(masks, confidences):\n        m_clipped = apply_solar_limb_mask(m.copy(), r_frac=r_frac)\n        area = int(m_clipped.sum())\n        if area >= min_area:\n            items.append({"mask": m_clipped, "conf": float(c), "area": area})\n\n    if not items:\n        return [], []\n\n    # Sort descending by confidence, then area\n    items.sort(key=lambda x: (x["conf"], x["area"]), reverse=True)\n\n    h, w = items[0]["mask"].shape[:2]\n    occupied = np.zeros((h, w), dtype=np.uint8)\n    sanitized_masks = []\n    sanitized_confs = []\n\n    for it in items:\n        m = it["mask"]\n        # Carve out already occupied pixels\n        m[occupied > 0] = 0\n        carved_area = int(m.sum())\n\n        if carved_area >= min_area:\n            occupied |= m\n            sanitized_masks.append(m)\n            sanitized_confs.append(it["conf"])\n\n    # Strict Zero-Overlap Assertion per Host Rule\n    n_kept = len(sanitized_masks)\n    for i in range(n_kept):\n        for j in range(i + 1, n_kept):\n            shared = int(np.logical_and(sanitized_masks[i], sanitized_masks[j]).sum())\n            if shared > 0:\n                raise AssertionError(f"FATAL: Overlap sanitizer failed! {shared} shared pixels between mask {i} and {j}.")\n\n    return sanitized_masks, sanitized_confs\n\n\ndef run_native_inference_on_disk(\n    model,\n    img_path: Path,\n    imgsz: int = MoonshotConfig.IMGSZ,\n    conf: float = MoonshotConfig.CONF,\n    nms_iou: float = MoonshotConfig.NMS_IOU,\n    min_area: int = MoonshotConfig.MIN_AREA,\n    device: str = "0",\n) -> Tuple[List[np.ndarray], List[float]]:\n    """Run native YOLO-seg inference on a single 2048x2048 image and sanitize."""\n    preds = model.predict(\n        source=str(img_path),\n        imgsz=imgsz,\n        conf=conf,\n        iou=nms_iou,\n        max_det=MoonshotConfig.MAX_DET,\n        device=device,\n        verbose=False,\n    )\n\n    pred_masks = []\n    pred_confs = []\n\n    if preds and preds[0].masks is not None:\n        raw_masks = preds[0].masks.data.cpu().numpy()\n        raw_confs = preds[0].boxes.conf.cpu().numpy()\n\n        for rm, c in zip(raw_masks, raw_confs):\n            # Upscale mask to native 2048x2048 if predicted at smaller imgsz\n            if rm.shape[:2] != (2048, 2048):\n                m_2048 = cv2.resize(rm.astype(np.uint8), (2048, 2048), interpolation=cv2.INTER_NEAREST)\n            else:\n                m_2048 = rm.astype(np.uint8)\n\n            if m_2048.sum() > 0:\n                pred_masks.append(m_2048)\n                pred_confs.append(float(c))\n\n    # Apply greedy zero-overlap sanitizer\n    clean_masks, clean_confs = sanitize_instances_zero_overlap(\n        pred_masks, pred_confs, min_area=min_area\n    )\n    return clean_masks, clean_confs\n\n\ndef generate_submission(\n    model,\n    test_dir: Path,\n    out_csv: Path,\n    imgsz: int = MoonshotConfig.IMGSZ,\n    conf: float = MoonshotConfig.CONF,\n    nms_iou: float = MoonshotConfig.NMS_IOU,\n    min_area: int = MoonshotConfig.MIN_AREA,\n    device: str = "0",\n    assert_180: bool = True,\n) -> pd.DataFrame:\n    """Generate official Kaggle submission CSV for all test images."""\n    import hashlib\n    import json\n\n    test_images = sorted(list(test_dir.glob("*.jpeg")) + list(test_dir.glob("*.jpg")))\n    print("=" * 75)\n    print("MOONSHOT 2048: NATIVE TEST INFERENCE & SUBMISSION BUILD (DIRECTIVE 18)")\n    print("=" * 75)\n    print(f"  Test Images Found:   {len(test_images)}")\n    print(f"  Inference imgsz:     {imgsz}")\n    print(f"  Confidence:          {conf}")\n    print(f"  NMS IoU:             {nms_iou}")\n    print(f"  Min Area:            {min_area} px")\n    print(f"  Output CSV:          {out_csv}")\n    print("=" * 75)\n\n    if assert_180:\n        assert len(test_images) == 180, f"FATAL: Expected exactly 180 test images, found {len(test_images)} in {test_dir}!"\n        print("  [DISCOVERY VERIFIED] Exactly 180 test images confirmed.")\n\n    rows = []\n    zero_pred_disks = 0\n    total_filaments = 0\n    processed_stems = []\n    zero_pred_stems = []\n\n    for idx, img_p in enumerate(test_images):\n        stem = img_p.stem\n        processed_stems.append(stem)\n        masks, confs = run_native_inference_on_disk(\n            model=model,\n            img_path=img_p,\n            imgsz=imgsz,\n            conf=conf,\n            nms_iou=nms_iou,\n            min_area=min_area,\n            device=device,\n        )\n\n        if not masks:\n            zero_pred_disks += 1\n            zero_pred_stems.append(stem)\n            # Zero-prediction rule: emit 0 rows for this disk\n            continue\n\n        for inst_idx, mask in enumerate(masks):\n            rle_str = encode_mask(mask)\n            filament_id = f"{stem}_{inst_idx}"\n            rows.append({\n                "filament_id": filament_id,\n                "segmentation_rle": rle_str,\n            })\n            total_filaments += 1\n\n        if (idx + 1) % 20 == 0 or (idx + 1) == len(test_images):\n            print(f"  [{idx + 1}/{len(test_images)}] processed | Filaments so far: {total_filaments}")\n\n    df_sub = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])\n    out_csv.parent.mkdir(parents=True, exist_ok=True)\n    df_sub.to_csv(out_csv, index=False)\n\n    sha256 = hashlib.sha256(out_csv.read_bytes()).hexdigest()\n\n    # Record inference manifest\n    manifest = {\n        "status": "SUCCESS",\n        "total_test_images": len(test_images),\n        "total_filaments_predicted": total_filaments,\n        "represented_disks": len(test_images) - zero_pred_disks,\n        "zero_detection_disks": zero_pred_disks,\n        "test_stems": processed_stems,\n        "zero_pred_stems": zero_pred_stems,\n        "submission_path": str(out_csv.resolve()),\n        "submission_sha256": sha256,\n        "inference_imgsz": imgsz,\n        "conf": conf,\n        "nms_iou": nms_iou,\n        "min_area": min_area,\n    }\n    manifest_path = out_csv.parent / "inference_manifest.json"\n    with open(manifest_path, "w", encoding="utf-8") as f:\n        json.dump(manifest, f, indent=2)\n\n    print("\\n" + "=" * 75)\n    print("SUBMISSION SUMMARY & MANIFEST:")\n    print(f"  Total Processed Images: {len(test_images)}")\n    print(f"  Total Submitted Rows:   {len(df_sub)}")\n    print(f"  Zero-Detection Disks:   {zero_pred_disks} (emitted 0 rows per host rule)")\n    print(f"  Wrote Submission To:    {out_csv} (SHA256: {sha256})")\n    print(f"  Wrote Manifest To:      {manifest_path}")\n    print("=" * 75)\n    return df_sub\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description="Native-2048 YOLO Inference")\n    parser.add_argument("--weights", type=str, default=str(MoonshotConfig.V8L_WEIGHTS_PATH))\n    parser.add_argument("--test-dir", type=str, default=str(MAGFILO_DIR / "test" / "test_images"))\n    parser.add_argument("--out", type=str, default=str(MoonshotConfig.SUBMISSION_PATH))\n    parser.add_argument("--imgsz", type=int, default=MoonshotConfig.IMGSZ)\n    parser.add_argument("--conf", type=float, default=MoonshotConfig.CONF)\n    parser.add_argument("--iou", type=float, default=MoonshotConfig.NMS_IOU)\n    parser.add_argument("--min-area", type=int, default=MoonshotConfig.MIN_AREA)\n    parser.add_argument("--device", type=str, default="0" if torch.cuda.is_available() else "cpu")\n    parser.add_argument("--dry-run", action="store_true")\n    args = parser.parse_args()\n\n    if args.dry_run:\n        print("[DRY-RUN] Native-2048 inference configured successfully. Exiting 0.")\n        sys.exit(0)\n\n    from ultralytics import YOLO\n    model = YOLO(args.weights)\n    generate_submission(\n        model=model,\n        test_dir=Path(args.test_dir),\n        out_csv=Path(args.out),\n        imgsz=args.imgsz,\n        conf=args.conf,\n        nms_iou=args.iou,\n        min_area=args.min_area,\n        device=args.device,\n    )\n')
with open("moonshot_2048/audit_submission.py", "w", encoding="utf-8") as f: f.write('"""\nmoonshot_2048/audit_submission.py — Strict Submission Contract Verifier.\n\nAuthority: ChatGPT Master\nDirective: 17 — Native-2048 Moonshot Toward 0.60 PQ\nExecutor: Antigravity\n\nMandatory Submission Contract Checks (Directive 17 Section 10):\n1. Exactly 180 test images accounted for.\n2. Zero-area masks == 0 (each row must decode to sum > 0).\n3. Invalid-RLE count == 0.\n4. Wrong-shape count == 0 (must decode to exactly (2048, 2048)).\n5. Strictly zero pairwise shared pixels per disk: sum(mask_i & mask_j) == 0.\n6. Duplicate filament_id count == 0.\n7. Disks with zero predictions emit zero rows (never dummy zero masks).\n8. Compute distribution of rows/image and mask areas, and SHA256.\n"""\n\nimport argparse\nimport hashlib\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom typing import Dict, Any, List\n\nimport numpy as np\nimport pandas as pd\nfrom metrics.pq import decode_rle\n\n\ndef audit_submission_csv(\n    csv_path: Path,\n    expected_test_stems: int = 180,\n    expected_shape: tuple = (2048, 2048),\n) -> Dict[str, Any]:\n    """Execute complete forensic contract audit on a submission CSV file."""\n    print("=" * 75)\n    print("MOONSHOT 2048: SUBMISSION CONTRACT AUDIT")\n    print("=" * 75)\n    print(f"  Target File: {csv_path}")\n\n    if not csv_path.exists():\n        raise FileNotFoundError(f"Submission file does not exist at {csv_path}")\n\n    raw_bytes = csv_path.read_bytes()\n    sha256_hash = hashlib.sha256(raw_bytes).hexdigest()\n    print(f"  SHA256: {sha256_hash}")\n\n    df = pd.read_csv(csv_path)\n    print(f"  Total Rows: {len(df)}")\n    print(f"  Columns: {list(df.columns)}")\n\n    # 1. Column contract\n    assert list(df.columns) == ["filament_id", "segmentation_rle"], \\\n        f"Column mismatch! Expected [\'filament_id\', \'segmentation_rle\'], got {list(df.columns)}"\n\n    # 2. Duplicate filament_id check\n    dup_ids = df[df.duplicated(subset=["filament_id"])]\n    n_dups = len(dup_ids)\n    print(f"  Duplicate filament_ids: {n_dups}")\n    assert n_dups == 0, f"FATAL: Found {n_dups} duplicate filament_ids!"\n\n    # 3. Disks representation\n    # filament_id format is typically \'{stem}_{idx}\'\n    stem_to_rows = defaultdict(list)\n    for idx, row in df.iterrows():\n        fid = str(row["filament_id"])\n        parts = fid.rsplit("_", 1)\n        stem = parts[0]\n        stem_to_rows[stem].append(row["segmentation_rle"])\n\n    represented_stems = len(stem_to_rows)\n    print(f"  Represented Disks: {represented_stems} (out of {expected_test_stems})")\n    assert represented_stems <= expected_test_stems, \\\n        f"FATAL: Represented stems ({represented_stems}) exceeds expected test stems ({expected_test_stems})!"\n\n    # 4. Check inference manifest and/or test directory if available\n    manifest_p = csv_path.parent / "inference_manifest.json"\n    if manifest_p.exists():\n        import json\n        with open(manifest_p, "r", encoding="utf-8") as mf:\n            man = json.load(mf)\n        assert man.get("total_test_images") == expected_test_stems, \\\n            f"FATAL: Manifest total_test_images ({man.get(\'total_test_images\')}) != {expected_test_stems}!"\n        assert man.get("represented_disks") == represented_stems, \\\n            f"FATAL: Manifest represented_disks ({man.get(\'represented_disks\')}) != {represented_stems}!"\n        print(f"  [MANIFEST VERIFIED] Verified against inference_manifest.json ({expected_test_stems} test images).")\n\n    # If test directory is discoverable, assert all represented stems are legitimate test stems\n    candidate_test_dirs = [\n        csv_path.parent / "test_images",\n        Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images"),\n        Path("/kaggle/input/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images"),\n        Path("data/MAGFiLO_1.0_Kaggle_2026/test/test_images"),\n    ]\n    discovered_test_dir = next((td for td in candidate_test_dirs if td.exists()), None)\n    if discovered_test_dir and expected_test_stems == 180:\n        found_test_files = list(discovered_test_dir.glob("*.jpeg")) + list(discovered_test_dir.glob("*.jpg"))\n        assert len(found_test_files) == 180, f"FATAL: Discovered test dir has {len(found_test_files)} images, expected 180!"\n        test_stem_set = {p.stem for p in found_test_files}\n        unknown_stems = set(stem_to_rows.keys()) - test_stem_set\n        assert len(unknown_stems) == 0, f"FATAL: Found {len(unknown_stems)} unknown stems not in test set: {list(unknown_stems)[:3]}"\n        print(f"  [TEST SET VERIFIED] All {represented_stems} represented stems belong to official 180 test images.")\n    zero_area_count = 0\n    invalid_rle_count = 0\n    wrong_shape_count = 0\n    pairwise_overlap_count = 0\n    mask_areas = []\n\n    print("  Decoding masks & verifying zero-overlap contract...")\n    for stem, rles in stem_to_rows.items():\n        disk_masks = []\n        for rle_str in rles:\n            try:\n                m = decode_rle(rle_str, h=expected_shape[0], w=expected_shape[1])\n            except Exception as e:\n                invalid_rle_count += 1\n                continue\n\n            if m.shape != expected_shape:\n                wrong_shape_count += 1\n\n            area = int(m.sum())\n            if area <= 0:\n                zero_area_count += 1\n            else:\n                mask_areas.append(area)\n                disk_masks.append(m)\n\n        # Assert zero pairwise overlap on this disk\n        n_m = len(disk_masks)\n        for i in range(n_m):\n            for j in range(i + 1, n_m):\n                shared = int(np.logical_and(disk_masks[i], disk_masks[j]).sum())\n                if shared > 0:\n                    pairwise_overlap_count += 1\n\n    # Print distribution stats\n    rows_per_disk = [len(rles) for rles in stem_to_rows.values()]\n\n    report = {\n        "csv_path": str(csv_path),\n        "sha256": sha256_hash,\n        "total_rows": len(df),\n        "represented_disks": represented_stems,\n        "expected_test_stems": expected_test_stems,\n        "duplicate_filament_ids": n_dups,\n        "zero_area_count": zero_area_count,\n        "invalid_rle_count": invalid_rle_count,\n        "wrong_shape_count": wrong_shape_count,\n        "pairwise_overlap_violations": pairwise_overlap_count,\n        "rows_per_disk_stats": {\n            "min": int(np.min(rows_per_disk)) if rows_per_disk else 0,\n            "mean": float(np.mean(rows_per_disk)) if rows_per_disk else 0.0,\n            "median": float(np.median(rows_per_disk)) if rows_per_disk else 0.0,\n            "max": int(np.max(rows_per_disk)) if rows_per_disk else 0,\n        },\n        "mask_area_stats": {\n            "min": int(np.min(mask_areas)) if mask_areas else 0,\n            "q25": float(np.percentile(mask_areas, 25)) if mask_areas else 0.0,\n            "median": float(np.median(mask_areas)) if mask_areas else 0.0,\n            "q75": float(np.percentile(mask_areas, 75)) if mask_areas else 0.0,\n            "max": int(np.max(mask_areas)) if mask_areas else 0,\n        },\n        "contract_passed": (\n            n_dups == 0\n            and zero_area_count == 0\n            and invalid_rle_count == 0\n            and wrong_shape_count == 0\n            and pairwise_overlap_count == 0\n        ),\n    }\n\n    print("\\n" + "=" * 75)\n    print("AUDIT RESULTS SUMMARY:")\n    print(f"  Zero-area masks:               {zero_area_count} (PASS if 0)")\n    print(f"  Invalid RLE strings:           {invalid_rle_count} (PASS if 0)")\n    print(f"  Wrong-shape masks:             {wrong_shape_count} (PASS if 0)")\n    print(f"  Pairwise overlap violations:   {pairwise_overlap_count} (PASS if 0)")\n    print(f"  Rows/disk distribution:        mean={report[\'rows_per_disk_stats\'][\'mean\']:.1f}, median={report[\'rows_per_disk_stats\'][\'median\']:.1f}, max={report[\'rows_per_disk_stats\'][\'max\']}")\n    print(f"  Mask area distribution (px):   median={report[\'mask_area_stats\'][\'median\']:.0f}, min={report[\'mask_area_stats\'][\'min\']}, max={report[\'mask_area_stats\'][\'max\']}")\n    print(f"  CONTRACT PASSED:               {report[\'contract_passed\']}")\n    print("=" * 75)\n\n    assert report["contract_passed"], f"FATAL: Submission failed contract checks! {report}"\n    return report\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description="Audit Submission CSV Contract")\n    parser.add_argument("csv_path", type=str)\n    args = parser.parse_args()\n\n    audit_submission_csv(Path(args.csv_path))\n')
print("✅ Deployed inference modules.")


In [ ]:
# ==============================================================================
# CELL 5: Run Native-2048 Inference & Generate submission.csv
# ==============================================================================
import time
from pathlib import Path
from ultralytics import YOLO
from moonshot_2048.predict_native import generate_submission
from moonshot_2048.config import MoonshotConfig

t0_inf = time.perf_counter()
model = YOLO(str(model_weights))
out_csv = Path("/kaggle/working/submission.csv")

# Operating point locked by Stage 2 Holdout Sweep winner (pq_mean=0.4583, pq_max=0.4845)
df_sub = generate_submission(
    model=model,
    test_dir=test_dir,
    out_csv=out_csv,
    imgsz=MoonshotConfig.IMGSZ,
    conf=0.25,
    nms_iou=0.00,
    min_area=50,
    device="0",
    assert_180=True,
)
print(f"Inference completed in {(time.perf_counter() - t0_inf):.1f} seconds.")


In [ ]:
# ==============================================================================
# CELL 6: Mandatory Submission Contract Audit
# ==============================================================================
from pathlib import Path
from moonshot_2048.audit_submission import audit_submission_csv

sub_path = Path("/kaggle/working/submission.csv")
audit_report = audit_submission_csv(
    csv_path=sub_path,
    expected_test_stems=180,
    expected_shape=(2048, 2048),
)
print("AUDIT STATUS: FULLY PASSED ALL HOST CONTRACT RULES")
